In [ ]:
#Version 1 of water data visualisation 
# CRS: ING EPSG: 29902 

import pandas as pd 
import numpy as np
import geopandas as gpd 

import leafmap
import matplotlib.pyplot as plt 
from scipy.stats import gaussian_kde
from shapely.geometry import Point


### Section 1 ###

## Loading and transforming data ## 

#Convert water data and stations into pandas dataframes 
water_data_raw = pd.read_csv('/Users/emmet/Desktop/WaterData/water_all_data.csv')
stations = pd.read_csv('/Users/emmet/Desktop/WaterData/stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')

rivers = rivers.to_crs(epsg=29902)

#rename columns to allow for later join
water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 

water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")


# print(stations.head())
# print(water_data.head())


#Create point objects from Lat/Long columns 
geometry = [Point(xy) for xy in zip(stations["Longitude"], stations["Latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")
geo_stations = geo_stations.to_crs(epsg=29902)

ph_data = water_data[(water_data["ParameterName"] == "pH") &
                     (water_data["SampleDate"].dt.year == 2019) &
                     (water_data["SampleDate"].dt.month == 6)]


#Outer Join (Merge) of two datasets for georeferenced chemistry data 
ph_data = ph_data.merge(geo_stations, on="StationID")
ph_median = ph_data.groupby("StationID").agg({
    "Result": "median", 
    "Latitude": "first", 
    "Longitude": "first"
}).reset_index()


# KDE Heatmap
x = ph_median["Longitude"].values
y = ph_median["Latitude"].values
z = ph_median["Result"].values

# Compute KDE
xy = np.vstack([x, y])
kde = gaussian_kde(xy, weights=z)
xmin, xmax = min(x), max(x)
ymin, ymax = min(y), max(y)
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 100), np.linspace(ymin, ymax, 100))
z_kde = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

# Leaflet map
m = leafmap.Map(center=[np.mean(y), np.mean(x)], zoom=10)
m.add_heatmap(ph_median, x="Longitude", y="Latitude", value="Result", radius=25)
m

# dodder_040 = full_data[full_data["EntityName"] == "DODDER"]

#print(full_data.head())

# Create buffer around river polyline 
# rivers_buffer50 = rivers.copy()
# rivers_buffer50["geometry"] = rivers_buffer50.geometry.buffer(50)  # 50m buffer

#print(rivers_buffer50.head())

#Dataframes now in use:

#Geo-referenced Chemistry Data - full_data
#River Buffers - rivers_buffer50

# m = leafmap.Map(center=[53.5, -7.5], zoom=7)

# #m.add_gdf(rivers, layer_name="50m River Buffer", style={"color": "green", "fillOpacity": 0.4})


# m.add_gdf(dodder_040, layer_name="Monitoring Stations", style={"color": "red", "radius": 5})


# m

In [25]:
#Version 1 of water data visualisation 
# CRS: ING EPSG: 29902 

import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
import matplotlib.pyplot as plt 


### Section 1 ###

## Loading and transforming data ## 

#Convert water data and stations into pandas dataframes 

water_data_raw = pd.read_csv('/Users/emmet/Desktop/WaterData/water_all_data.csv')
stations = pd.read_csv('/Users/emmet/Desktop/WaterData/dodder_stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')


#rename columns to allow for later join
water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 

# Convert SampleDate to Time format
water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")


#Create point objects from Lat/Long columns 
geometry = [Point(xy) for xy in zip(stations["Longitude"], stations["Latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")
geo_stations = geo_stations.to_crs(epsg=29902)


#Outer Join (Merge) of two datasets for georeferenced chemistry data 
georef_do_data = geo_stations.merge(water_data, on="StationID", how='inner')

do_june = water_data[(water_data["ParameterName"] == "pH") &
                     (water_data["SampleDate"].dt.year == 2019) &
                     (water_data["SampleDate"].dt.month == 6)]


# Merge pH data with station locations to get coordinates
do_june = do_june.merge(geo_stations[["StationID", "Latitude", "Longitude"]], on="StationID", how="left")

# Compute median pH per station
ph_median = do_june.groupby("StationID").agg({
    "Result": "median", 
    "Latitude": "first", 
    "Longitude": "first"
}).reset_index()




In [32]:
import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
import matplotlib.pyplot as plt 

### Section 1 ###

## Loading and transforming data ## 

# Convert water data and stations into pandas dataframes 
water_data_raw = pd.read_csv('/Users/emmet/Desktop/WaterData/water_all_data.csv')
stations = pd.read_csv('/Users/emmet/Desktop/WaterData/dodder_stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')
rivers = rivers.to_crs(epsg=4326)

# Rename columns to allow for later join
water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 

# Convert SampleDate to Time format
water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")

# Create point objects from Lat/Long columns 
geometry = [Point(xy) for xy in zip(stations["Longitude"], stations["Latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")

# Outer Join (Merge) of two datasets for georeferenced chemistry data 
georef_do_data = geo_stations.merge(water_data, on="StationID", how='inner')

do_june = water_data[(water_data["ParameterName"] == "pH") &
                     (water_data["SampleDate"].dt.year == 2019) &
                     (water_data["SampleDate"].dt.month == 6)]

# Merge pH data with station locations to get coordinates
do_june = do_june.merge(geo_stations[["StationID", "Latitude", "Longitude"]], on="StationID", how="left")
do_june = do_june.rename(columns={"Latitude": "latitude", "Longitude": "longitude"})
# Compute median pH per station
ph_median = do_june.groupby("StationID").agg({
    "Result": "median", 
    "Latitude": "first", 
    "Longitude": "first"
}).reset_index()

# Create an interactive Leaflet map
m = leafmap.Map(center=[53.3, -6.3], zoom=12)

# Add river layer
m.add_gdf(rivers, layer_name="Rivers")

# Add pH median heatmap
m.add_heatmap(ph_median, x="longitude", y="latitude", value="Result", radius=25, name="pH Median Heatmap")

# Show the map
m


KeyError: "Column(s) ['Latitude', 'Longitude'] do not exist"